# QITE — Quantum Imaginary-Time Evolution

Motta-style QITE drives an initial state toward the ground state by imaginary-time evolution, with **no classical optimizer**. Each step solves a small real linear system for a Pauli generator and applies it as a `TrotterBlock`, accumulating a state circuit.

In [ ]:
import numpy as np

from qarp.algorithms import QITE
from qarp.blocks import ComputationalBasisStateBlock
from qarp.operators.models import transverse_field_ising

## A spin model: transverse-field Ising

A quick, dependency-free start on a 2-qubit TFIM.

In [ ]:
H = transverse_field_ising((2,), j=1.0, h_x=0.8)
initial_block = ComputationalBasisStateBlock([0, 0])

qite = QITE(H, initial_block, dtau=0.05, n_steps=200)
energy, psi = qite.run()

exact = float(np.linalg.eigvalsh(H.sparse_matrix(2).toarray())[0])
print(f"QITE energy: {energy:.6f}   exact: {exact:.6f}")

The per-step energies trace the imaginary-time trajectory (monotonically non-increasing), and the optimized state is a re-simulable circuit via `get_final_state_block`.

In [ ]:
print("initial / final energy:", qite.energy_history[0], qite.energy_history[-1])

final_block = qite.get_final_state_block()
print("final-state block, qubits:", final_block.n_qubits)

## Quantum chemistry: H₂

The same driver on a real molecular Hamiltonian: H₂ in the STO-3G basis (4 qubits), built from a pyscf mean field and Jordan-Wigner mapped. QITE from the Hartree–Fock reference reaches the FCI energy — the same `pyscf → restricted_integrals_to_fermion_operator → JordanWigner` pattern the VQE / QSE examples use.

In [ ]:
from pyscf import gto, scf, ao2mo

from qarp.operators import JordanWigner
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.onv import onv_from_spatial_occupations

mol = gto.M(atom="H 0 0 0; H 0 0 0.74", basis="sto3g")
mol.build()
mf = scf.RHF(mol)
mf.kernel()

constant = mf.energy_nuc()
h1 = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
h2 = ao2mo.full(mf.mol, mf.mo_coeff, aosym="s1").reshape([h1.shape[0]] * 4)
qham = JordanWigner().encode_operator(
    restricted_integrals_to_fermion_operator(constant, h1, h2)
)
onv = onv_from_spatial_occupations([2, 0])  # Hartree-Fock reference (2 electrons)

qite_h2 = QITE(qham, ComputationalBasisStateBlock(onv), dtau=0.1, n_steps=50)
energy_h2, _ = qite_h2.run()

fci = float(np.linalg.eigvalsh(qham.sparse_matrix(4).toarray())[0])
print(f"QITE H2 energy: {energy_h2:.6f}   FCI: {fci:.6f}   gap: {energy_h2 - fci:.2e}")